In [1]:
# import library
import pandas as pd
import ast
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast
import torch
from torch.utils.data import Dataset
from transformers import BertForSequenceClassification
from torch.utils.data import DataLoader
from transformers import AdamW
from tqdm import tqdm
import numpy as np
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

/opt/anaconda3/envs/ARP/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("data/ARP_dataset_fixed_Sentiment.csv")

def extract_entity_names(entities_str):
    try:
        entities = ast.literal_eval(entities_str)
        return [e[0] for e in entities if isinstance(e, tuple)]
    except:
        return []

df["Entity_Texts"] = df["Entities"].apply(extract_entity_names)


In [3]:
ENTITY_LIST = [
    "Federal Reserve", "Interest Rates", "Inflation", "Employment", "Unemployment", "GDP", "Trade", "Congress", "Monetary Policy", "Financial Stability", 
    "Price Stability", "Regulatory Implementation", "Pandemic", "Asset Runoff", "Reinvestment", "Money Market", "Bond Market", "Equity Markets", "Financial Markets", "Repo Markets", 
    "Fiscal Policy", "Balance Sheet", "Reserves", "Digital Dollar", "Foreign Currencies", "Federal Funds", "Demand", "Securities", "War", "Finance", 
    "Debt", "Mortgage", "Maturity", "Credit", "Labor Market", "Auction", "Press Conference", "Banking System", "Uncertain", "Development", "Economic Outlook", "Countries"
]


In [4]:
# label vector (0 or 1) for each sentence
def label_vector_from_entities(entity_names):
    vec = [0] * len(ENTITY_LIST)
    for i, ent in enumerate(ENTITY_LIST):
        if ent in entity_names:
            vec[i] = 1
    return vec

df["Label_Vector"] = df["Entity_Texts"].apply(label_vector_from_entities)


In [6]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=123)

In [7]:
# converted to strings
train_df["Sentence"] = train_df["Sentence"].astype(str)
test_df["Sentence"]  = test_df["Sentence"].astype(str)

In [8]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")

# encode
train_encodings = tokenizer(train_df["Sentence"].tolist(), truncation=True, padding=True, max_length=128)
test_encodings  = tokenizer(test_df["Sentence"].tolist(),  truncation=True, padding=True, max_length=128)

train_labels = train_df["Label_Vector"].tolist()
test_labels  = test_df["Label_Vector"].tolist()


In [9]:

class MultiLabelDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        return {
            key: torch.tensor(val[idx]) for key, val in self.encodings.items()
        } | {
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

    def __len__(self):
        return len(self.labels)

train_dataset = MultiLabelDataset(train_encodings, train_labels)
test_dataset  = MultiLabelDataset(test_encodings, test_labels)


In [10]:

model = BertForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(ENTITY_LIST),
    problem_type="multi_label_classification"
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
val_ratio  = 0.1       
seed       = 123          
all_idx    = np.arange(len(train_dataset))
train_idx, val_idx = train_test_split(
    all_idx, test_size=val_ratio, random_state=seed, shuffle=True
)

train_subset = Subset(train_dataset, train_idx)
val_subset   = Subset(train_dataset, val_idx)

# DataLoader（train: shuffle=True；val/test: shuffle=False）
train_loader = DataLoader(train_subset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_subset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
# preparing the training data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)

# calculate pos_weight
label_matrix = np.array(train_labels)  # shape: [num_samples, num_labels]
pos_counts = label_matrix.sum(axis=0)
neg_counts = len(label_matrix) - pos_counts
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float).to(device)

# initialise the weighted loss function
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# early stop 
num_epochs   = 100
patience     = 8          
min_delta    = 1e-4       
best_metric  = -1.0       
epochs_no_improve = 0
best_state   = None

/opt/anaconda3/envs/ARP/lib/python3.12/site-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [13]:
# trainning process
model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    print(f"Epoch {epoch+1}/{num_epochs}")

    model.train()
    for batch in tqdm(train_loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) 
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # use Micro-F1 as an early stop indicator
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_np      = batch["labels"].cpu().numpy()

            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs > 0.5).astype(int)  

            all_preds.append(preds)
            all_labels.append(labels_np)

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_labels)

    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"Train Avg loss: {avg_loss:.4f} | Val Micro-F1: {micro_f1:.4f} | Val Macro-F1: {macro_f1:.4f}")

    if micro_f1 - best_metric > min_delta:
        best_metric = micro_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        torch.save(best_state, "best_state.pt")
        print(f"New best Micro-F1={best_metric:.4f} — checkpoint updated")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve}/{patience} epoch(s)")
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

# restore/save best model
if best_state is None:
    try:
        best_state = torch.load("best_state.pt", map_location="cpu")
    except:
        pass

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)
    model.eval()
    model.save_pretrained("./best_model_entities")
    tokenizer.save_pretrained("./best_model_entities")
    print(f"Best model saved to ./best_model_entities (Val Micro-F1={best_metric:.4f})")
else:
    print("Best model not found; retain last-round model")


Epoch 1/100


100%|██████████| 53/53 [01:30<00:00,  1.70s/it]


Train Avg loss: 1.3330 | Val Micro-F1: 0.1397 | Val Macro-F1: 0.1084
New best Micro-F1=0.1397 — checkpoint updated
Epoch 2/100


100%|██████████| 53/53 [01:34<00:00,  1.78s/it]


Train Avg loss: 1.1817 | Val Micro-F1: 0.1674 | Val Macro-F1: 0.1278
New best Micro-F1=0.1674 — checkpoint updated
Epoch 3/100


...

Train Avg loss: 0.0359 | Val Micro-F1: 0.7193 | Val Macro-F1: 0.3850
No improvement for 6/8 epoch(s)
Epoch 71/100


100%|██████████| 53/53 [01:22<00:00,  1.55s/it]


Train Avg loss: 0.0357 | Val Micro-F1: 0.7366 | Val Macro-F1: 0.4145
No improvement for 7/8 epoch(s)
Epoch 72/100


100%|██████████| 53/53 [01:22<00:00,  1.55s/it]


Train Avg loss: 0.0338 | Val Micro-F1: 0.7424 | Val Macro-F1: 0.3949
No improvement for 8/8 epoch(s)
Early stopping triggered at epoch 72
Best model saved to ./best_model_entities (Val Micro-F1=0.7473)


In [ ]:
#model.save_pretrained(f"best_model_entities")
#tokenizer.save_pretrained(f"best_model_entities")

Test Data

In [14]:
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import numpy as np
from tqdm import tqdm


model_path = "best_model_entities" 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

tokenizer = BertTokenizerFast.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()


all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()  

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()
        preds = (probs > 0.5).astype(int) 

        all_preds.append(preds)
        all_labels.append(labels)


y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)

print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Precision (micro):", precision_score(y_true, y_pred, average="micro"))
print("Recall (micro):", recall_score(y_true, y_pred, average="micro"))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=ENTITY_LIST))

100%|██████████| 13/13 [00:07<00:00,  1.63it/s]

Micro F1: 0.7168709865732633
Macro F1: 0.5914609778422554
Precision (micro): 0.6602150537634408
Recall (micro): 0.7841634738186463

Classification Report:

                           precision    recall  f1-score   support

          Federal Reserve       0.88      0.80      0.84        87
           Interest Rates       0.69      0.82      0.75        50
                Inflation       0.89      0.86      0.87        93
               Employment       0.69      0.81      0.75        31
             Unemployment       1.00      1.00      1.00         8
                      GDP       0.60      0.68      0.64        31
                    Trade       1.00      1.00      1.00         3
                 Congress       0.50      0.33      0.40         3
          Monetary Policy       0.61      0.74      0.67        66
      Financial Stability       0.00      0.00      0.00         1
          Price Stability       0.62      0.64      0.63        25
Regulatory Implementation       0.00   


/opt/anaconda3/envs/ARP/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/ARP/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/ARP/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaco